# ASF catalog explorer

Explore the bundled ASF product catalog **without any network or credentials** — it ships as package data. The catalog covers Sentinel-1, ALOS PALSAR, OPERA-S1, ARIA GUNW, NISAR, plus the legacy ERS / JERS / RADARSAT / SEASAT / SIR-C / AIRSAR / UAVSAR / SMAP archives. See [Available products](../../reference/asf/datasets.md) for the rendered table and [Usage](../../reference/asf/usage.md) for the download walkthrough.

## Setup

We only need the `Catalog` loader. Constructing it parses the package-data YAML — no network call, no SDK import.

In [ ]:
from collections import Counter

from earthlens.asf import Catalog

cat = Catalog()

### Catalog at a glance

Two views are useful: the curated `products` (with full metadata) and the informational `available_products` index (just the sorted keys).

In [ ]:
print(f'curated products:         {len(cat.products)}')
print(f'available_products index: {len(cat.available_products)}')
print(f'stackable subset:         {len(cat.stackable_products())}')

## Stackable vs search-only

The defining classification: `stackable: true` rows are the products you can hand to `ASFProduct.stack()` for an InSAR baseline search. `stackable: false` rows are processed-derivative outputs (RTC, GUNW), analysis-ready amplitudes (GRD, OCN), or legacy archives whose product class isn't an `ASFStackableProduct`.

In [ ]:
print('Stackable (InSAR-ready):')
for key in cat.stackable_products():
    row = cat.get_product(key)
    print(f'  {key:24s} -> {row.platform or row.dataset:18s} {row.product_type}')

print('\nSearch-only:')
for key, row in cat.products.items():
    if not row.stackable:
        print(f'  {key:24s} -> {row.platform or row.dataset:18s} {row.product_type}')

## Coverage by platform / dataset

Each row carries **exactly one** of a `platform` (an `asf_search.PLATFORM` member) or a `dataset` (an `asf_search.DATASET` member, used for the processed-product families OPERA-S1, ARIA-S1-GUNW, SLC-BURST, ALOS-2). Grouping by that key shows which constellations dominate the catalog.

In [ ]:
by_platform = Counter(row.platform for row in cat.products.values() if row.platform)
by_dataset = Counter(row.dataset for row in cat.products.values() if row.dataset)

print('By platform:')
for k, n in sorted(by_platform.items()):
    print(f'  {k:14s} {n}')
print('\nBy dataset (processed families):')
for k, n in sorted(by_dataset.items()):
    print(f'  {k:14s} {n}')

## Aliases

Every curated key has a small set of friendly aliases (`s1-slc` → `sentinel-1-slc`, `palsar-slc` → `alos-palsar-slc`, …). The aliases are globally unique — the catalog's `model_post_init` invariant check rejects a YAML that reuses an alias across rows.

In [ ]:
for key, row in cat.products.items():
    if row.aliases:
        print(f'  {key:24s} aliases: {row.aliases}')

### Resolving an alias

`resolve()` accepts a curated key or any alias and returns the curated key. An unknown name raises with a did-you-mean hint.

In [ ]:
print(cat.resolve('s1-slc'))  # alias
print(cat.resolve('sentinel-1-slc'))  # canonical key
print(cat.resolve('opera-rtc'))  # alias
print(cat.resolve('alos-slc'))  # alias

## Picking a product

A short decision table:

| You want | Product key |
|---|---|
| C-band InSAR over land, 2014+ | `sentinel-1-slc` |
| Sub-AOI InSAR (smaller download) | `sentinel-1-burst` |
| L-band InSAR (forest / agriculture) | `alos-palsar-slc` |
| Pre-Sentinel baseline (1990s, 2000s) | `ers-1-slc`, `ers-2-slc`, `radarsat-1-l1` |
| Analysis-ready backscatter (no phase) | `opera-rtc-s1`, `sentinel-1-grd` |
| Precomputed interferogram | `aria-s1-gunw` |
| Disturbance flag | `opera-dist-alert-s1` |

See [Usage](../../reference/asf/usage.md) for the worked search-mode and stack-mode examples.